In [2]:
import pandas as pd
import numpy as np

In [8]:
spam = pd.read_csv("../data/cleaned/spam.csv")
spam.sample(5)

,target,text,count_alphabet,count_words,count_sent,preprocessed_text
3911,0,Haven't heard anything and he's not answering ...,111,24,2,heard anyth answer text guess flake said jb fa...
4172,0,Thanks again for your reply today. When is ur ...,236,55,5,thank repli today ur visa come r u still buy g...
4208,0,Nvm take ur time.,17,5,1,nvm take ur time
2691,0,Then u ask darren go n pick u lor... But i oso...,74,20,2,u ask darren go n pick u lor oso sian tmr haf ...
3546,0,Where are you call me.,22,6,1,call


In [9]:
print("Null count: ", spam["preprocessed_text"].isna().sum())
spam.dropna(inplace= True)
print(spam.isna().sum(), spam.shape)

Null count:  9
target               0
text                 0
count_alphabet       0
count_words          0
count_sent           0
preprocessed_text    0
dtype: int64 (5160, 6)


In [10]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

In [11]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score,confusion_matrix,precision_score

In [7]:
gnb = GaussianNB()
mnb = MultinomialNB()
bnb = BernoulliNB()
lr = LogisticRegression()
knn = KNeighborsClassifier()
dt = DecisionTreeClassifier()
svc = SVC()
rf = RandomForestClassifier(n_estimators=50, random_state=2, n_jobs=-1)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2, n_jobs= -1)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2, n_jobs=-1)
gbdt = GradientBoostingClassifier(n_estimators=50,random_state=2)

In [20]:
models = {
  gnb: "Gaussian Naive Bayes",
  mnb: "Mutinomial Naive Bayes",
  bnb: "Bernoulli Naive Bayes",
  lr : "Logistic Regression",
  knn : "K Neighbors Classifier",
  dt : "Decision Tree Classifier",
  rf : "Random Forest Classifier",
  svc : "Support Vector Classifier",
  abc : "Ada Boost Classifier",
  bc : "Bagging Classifier",
  etc : "Extra Trees Classifier",
  gbdt : "Gradient Boosting Classifier",
}

models

{GaussianNB(): 'Gaussian Naive Bayes',
 MultinomialNB(): 'Mutinomial Naive Bayes',
 BernoulliNB(): 'Bernoulli Naive Bayes',
 LogisticRegression(): 'Logistic Regression',
 KNeighborsClassifier(): 'K Neighbors Classifier',
 DecisionTreeClassifier(): 'Decision Tree Classifier',
 RandomForestClassifier(max_depth=50, min_samples_leaf=9, n_estimators=35,
                        n_jobs=-1): 'Random Forest Classifier',
 SVC(): 'Support Vector Classifier',
 AdaBoostClassifier(random_state=2): 'Ada Boost Classifier',
 BaggingClassifier(n_estimators=50, n_jobs=-1, random_state=2): 'Bagging Classifier',
 ExtraTreesClassifier(n_estimators=50, n_jobs=-1, random_state=2): 'Extra Trees Classifier',
 GradientBoostingClassifier(n_estimators=50, random_state=2): 'Gradient Boosting Classifier'}

In [21]:
from typing import Dict

def train_model(models: Dict[object, str], vectorizer: str = "count", features: int = 5000, log= True) -> pd.DataFrame:

  if vectorizer == "tfidf":
    tfidf = TfidfVectorizer(max_features= features, stop_words="english")
    X = tfidf.fit_transform(spam["preprocessed_text"]).toarray()
    y = spam["target"]

  else:
    cv = CountVectorizer(max_features= features)
    X = cv.fit_transform(spam["preprocessed_text"]).toarray()
    y = spam["target"]

  X_train, X_test, y_trian, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

  data = {
    "Model": [],
    f"Accuracy ({vectorizer}, feature= {features})": [],
    f"Precision ({vectorizer}, feature= {features})": [],
  }

  for model in list(models.keys()):
    if log:
      print(f"Training model: {models.get(model)}")
    m = model.fit(X_train, y_trian)
    y_pred = model.predict(X_test)

    data["Model"].append(models.get(model))
    data[f"Accuracy ({vectorizer}, feature= {features})"].append(accuracy_score(y_test, y_pred))
    data[f"Precision ({vectorizer}, feature= {features})"].append(precision_score(y_test, y_pred))

  return pd.DataFrame(data).sort_values(by= f"Precision ({vectorizer}, feature= {features})", ascending= False)

In [10]:
df1 = train_model(models)

Training model: Gaussian Naive Bayes
Training model: Mutinomial Naive Bayes
Training model: Bernoulli Naive Bayes
Training model: Logistic Regression
Training model: K Neighbors Classifier
Training model: Decision Tree Classifier
Training model: Random Forest Classifier
Training model: Support Vector Classifier
Training model: Ada Boost Classifier
Training model: Bagging Classifier
Training model: Extra Trees Classifier
Training model: Gradient Boosting Classifier


In [11]:
df2 = train_model(models, vectorizer="tfidf")

Training model: Gaussian Naive Bayes
Training model: Mutinomial Naive Bayes
Training model: Bernoulli Naive Bayes
Training model: Logistic Regression
Training model: K Neighbors Classifier
Training model: Decision Tree Classifier
Training model: Random Forest Classifier
Training model: Support Vector Classifier
Training model: Ada Boost Classifier
Training model: Bagging Classifier
Training model: Extra Trees Classifier
Training model: Gradient Boosting Classifier


In [12]:
df1.merge(df2, on="Model")

,Model,Accuracy count,Precision count,Accuracy tfidf,Precision tfidf
0,K Neighbors Classifier,0.926357,1.000000,0.918605,1.000000
1,Random Forest Classifier,0.974806,1.000000,0.973837,0.980000
2,Logistic Regression,0.973837,0.980000,0.951550,0.962025
3,Support Vector Classifier,0.972868,0.979798,0.971899,0.989583
4,Extra Trees Classifier,0.970930,0.979381,0.969961,0.979167
5,Bernoulli Naive Bayes,0.965116,0.957895,0.964147,0.957447
6,Gradient Boosting Classifier,0.954457,0.941860,0.950581,0.950000
7,Bagging Classifier,0.968023,0.916667,0.963178,0.869565
8,Mutinomial Naive Bayes,0.970930,0.866142,0.965116,1.000000
9,Ada Boost Classifier,0.922481,0.852459,0.915698,0.928571


In [32]:
### best max_feature == 3000
tfidf = TfidfVectorizer(max_features= 3500, stop_words="english")
X = tfidf.fit_transform(spam["preprocessed_text"]).toarray()
y = spam["target"]

X_train, X_test, y_trian, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
mnb = MultinomialNB()
mnb.fit(X_train, y_trian)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3598., 530.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.05]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 3500)","[[ 0. , 0.76, 0. ,..., 0.36, 1.23,16.49], [ 0.35, 0. , 0.38,..., 0. , 0. , 0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 3500)","[[-9.34,-8.78,-9.34,...,-9.03,-8.54,-6.48], [-8.26,-8.56,-8.24,...,-8.56,-8.56,-8.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,3500


In [34]:
y_pred = mnb.predict(X_test)
print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision: ", precision_score(y_test, y_pred))

Accuracy:  0.9718992248062015
Precision:  1.0


In [35]:
import pickle
pickle.dump(mnb, open("../models/model.pkl", "wb"))
pickle.dump(tfidf, open("../models/vectorizer.pkl", "wb"))